# positive_control_convert — FSDP checkpoints to loadable adapters

Same conversion as `lora_convert`, applied to the five checkpoints of `positive_control_r8`:
rebuild the adapter config, load the fused state dict, extract the LoRA layers via
`get_peft_model_state_dict`, write `pytorch_lora_weights.safetensors`. Both environment fixes
carry over — torchao removed so peft's dispatcher does not raise, and the module built on CPU
rather than under `init_empty_weights()`.

Section 4 adds one check the arm conversions could not make: the `lora_B` norm of each converted
adapter against the `[ctrl]` trace in `train_console.log` at the same step. The print in the
patched `train.py` fires before the checkpoint is written, with no optimiser step between, so the
two are the same tensor and must agree to float precision. A mismatch means a checkpoint and a
log line have been paired wrongly.

High-RAM runtime: the CPU model plus the mmap'd 15.9 GB checkpoint exceeds a standard runtime's
~12 GB. The A100 runtime (83 GB system RAM) or a High-RAM CPU runtime is fine.

## 1. Repo, environment fixes

In [1]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's LoRA injection walks every dispatcher, including dispatch_torchao, which raises rather than
# returning False when torchao predates 0.16 (Colab ships 0.10). torchao is unused by this bf16 path;
# removing it makes is_torchao_available() return False so the dispatcher is skipped.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

sys.path.insert(0, REPO)
os.chdir(REPO)

import peft, diffusers, transformers
print(f"peft {peft.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
assert tuple(int(x) for x in peft.__version__.split(".")[:2]) >= (0, 17), \
    "diffusers needs peft>=0.17; do not pin peft below it"
_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel;"
          "from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline;"
          "print('convert imports OK')")
_chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(_chk.stdout.strip() or _chk.stderr.strip()[-800:])
assert _chk.returncode == 0, "converter imports are broken"

installed.
torchao removed
peft 0.20.0 | diffusers 0.39.0 | transformers 4.51.3
convert imports OK


## 2. Drive, checkpoint inventory

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
EXP   = f"{DRIVE}/lora/experiments"
RUN   = "positive_control_r8"
STEPS = [100, 200, 300, 400, 500]

CFG = f"{EXP}/{RUN}/{RUN}.yml"
assert os.path.exists(CFG), f"missing config: {CFG}"
for s in STEPS:
    ckpt = f"{EXP}/{RUN}/checkpoint-{s}/pytorch_model_fsdp.bin"
    assert os.path.exists(ckpt), f"missing checkpoint: {ckpt}"
    print(f"step {s:>4d}: checkpoint present ({os.path.getsize(ckpt)/1e9:.2f} GB)")
print(f"\nconfig: {CFG}")

Mounted at /content/drive
step  100: checkpoint present (15.89 GB)
step  200: checkpoint present (15.89 GB)
step  300: checkpoint present (15.89 GB)
step  400: checkpoint present (15.89 GB)
step  500: checkpoint present (15.89 GB)

config: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/positive_control_r8/positive_control_r8.yml


## 3. Convert — fused checkpoints to adapters

In [3]:
# Reproduces convert_ckpt_to_hf_format.py, but constructs the module on CPU rather than under
# init_empty_weights(), where the current peft cannot size LoRA layers on meta tensors.
# load_state_dict(assign=True) then replaces every base and adapter tensor with the checkpoint's
# values, so the extracted adapter is identical to the script's intent.
import torch, gc
from omegaconf import OmegaConf
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline

TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]

def convert(step):
    ckpt = f"{EXP}/{RUN}/checkpoint-{step}/pytorch_model_fsdp.bin"
    save = f"{EXP}/{RUN}/checkpoint-{step}/transformer_lora"
    if os.path.exists(f"{save}/pytorch_lora_weights.safetensors"):
        print(f"step {step}: adapter already present, skipped")
        return

    conf = OmegaConf.load(CFG)
    assert conf.train.get("lora_ft", False), "config is not a LoRA run"
    arch_opt = OmegaConf.to_object(conf.model.arch_opt)
    for k, v in arch_opt.items():
        if isinstance(v, list): arch_opt[k] = tuple(v)

    transformer = OmniGen2Transformer2DModel(**arch_opt)
    transformer.add_adapter(LoraConfig(
        r=conf.train.lora_rank, lora_alpha=conf.train.lora_rank,
        lora_dropout=conf.train.lora_dropout, init_lora_weights="gaussian",
        target_modules=TARGET_MODULES,
    ))

    state_dict = torch.load(ckpt, mmap=True, weights_only=True)
    missing, unexpect = transformer.load_state_dict(state_dict, assign=True, strict=False)
    lora_missing = [k for k in missing if "lora_" in k]
    assert not lora_missing, f"step {step}: adapter keys not filled from checkpoint: {lora_missing[:4]}"
    assert not unexpect,     f"step {step}: unexpected keys in checkpoint: {unexpect[:4]}"

    adapter = get_peft_model_state_dict(transformer)
    OmniGen2Pipeline.save_lora_weights(save_directory=save, transformer_lora_layers=adapter)
    n = sum(v.numel() for v in adapter.values())
    print(f"step {step}: {len(adapter)} adapter tensors, {n/1e6:.3f} M params -> {save}")

    del transformer, state_dict, adapter
    gc.collect()

for s in STEPS:
    print(f"\n===== converting step {s} =====")
    convert(s)
print("\nall conversions done")

/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



===== converting step 100 =====
step 100: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/positive_control_r8/checkpoint-100/transformer_lora

===== converting step 200 =====
step 200: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/positive_control_r8/checkpoint-200/transformer_lora

===== converting step 300 =====
step 300: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/positive_control_r8/checkpoint-300/transformer_lora

===== converting step 400 =====
step 400: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/positive_control_r8/checkpoint-400/transformer_lora

===== converting step 500 =====
step 500: 304 adapter tensors, 5.107 M params -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data

## 4. Verify adapters against the training log

In [4]:
# Each adapter must carry the rank-8 parameter count, and its lora_B norm must match the [ctrl]
# trace at the same step: the patched print fires before the checkpoint is written, with no
# optimiser step between, so the two read the same tensor.
import re
from safetensors.torch import load_file

LOG = f"{EXP}/{RUN}/train_console.log"
trace = dict((int(s), float(b)) for s, b in
             re.findall(r"\[ctrl\] step (\d+) b_norm (\S+)", open(LOG, encoding="utf-8").read())) \
        if os.path.exists(LOG) else {}
print(f"{'step':>6s} {'tensors':>8s} {'params':>12s} {'MB':>7s} {'||B|| adapter':>14s} {'||B|| log':>12s} {'diff':>10s}")
for s in STEPS:
    path = f"{EXP}/{RUN}/checkpoint-{s}/transformer_lora/pytorch_lora_weights.safetensors"
    assert os.path.exists(path), f"adapter not written: {path}"
    sd = load_file(path)
    n_params = sum(v.numel() for v in sd.values())
    bn = float(torch.cat([v.float().flatten() for k, v in sorted(sd.items()) if "lora_B" in k]).norm())
    lg = trace.get(s)
    diff = "-" if lg is None else f"{abs(bn-lg):.2e}"
    print(f"{s:>6d} {len(sd):>8d} {n_params:>12,d} {os.path.getsize(path)/1e6:>7.1f} "
          f"{bn:>14.4f} {'-' if lg is None else f'{lg:.4f}':>12s} {diff:>10s}")
    assert n_params == 5_107_200, f"step {s}: {n_params} params, expected 5,107,200 for rank 8"
    if lg is not None:
        assert abs(bn - lg) < 1e-3, f"step {s}: adapter ||B|| {bn} disagrees with the log's {lg}"

k = next(iter(sd))
print(f"\nexample key: {k}")
print(f"dtype: {sd[k].dtype}")
print("\nnext: positive_control_gen on the same 302 screen rows")

  step  tensors       params      MB  ||B|| adapter    ||B|| log       diff
   100      304    5,107,200    20.5         2.0502       2.0503   7.66e-05
   200      304    5,107,200    20.5         3.2397       3.2398   1.40e-04
   300      304    5,107,200    20.5         4.2500       4.2502   1.84e-04
   400      304    5,107,200    20.5         5.1760       5.1762   1.87e-04
   500      304    5,107,200    20.5         6.0275       6.0277   2.51e-04

example key: transformer.context_refiner.0.attn.to_k.lora_A.weight
dtype: torch.float32

next: positive_control_gen on the same 302 screen rows
